In [0]:
# ========================================
# CONFIGURATION
# ========================================
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
print("Configuration chargée")

Configuration chargée


In [0]:
# ========================================
# CHARGEMENT DONNÉES NETTOYÉES Chargement de la table Delta nettoyée
# ========================================
catalog = "workspace"
schema_name = "default"
table_clean = "hospital_readmissions_clean"
df = spark.table(f"{catalog}.{schema_name}.{table_clean}")
print(f"Dataset chargé : {df.count():,} lignes{len(df.columns)} colonnes")

Dataset chargé : 69,750 lignes25 colonnes


In [0]:
df.printSchema()

root
 |-- patient_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- genre: string (nullable = true)
 |-- diabete: boolean (nullable = true)
 |-- hypertension: boolean (nullable = true)
 |-- insuffisance_cardiaque: boolean (nullable = true)
 |-- maladie_renale: boolean (nullable = true)
 |-- cancer: boolean (nullable = true)
 |-- depression: boolean (nullable = true)
 |-- nb_comorbidites: integer (nullable = true)
 |-- charlson_index: integer (nullable = true)
 |-- diagnostic_principal: string (nullable = true)
 |-- duree_sejour_jours: integer (nullable = true)
 |-- admission_urgence: boolean (nullable = true)
 |-- service: string (nullable = true)
 |-- date_admission: timestamp (nullable = true)
 |-- date_sortie: timestamp (nullable = true)
 |-- saison: string (nullable = true)
 |-- proba_readmission: double (nullable = true)
 |-- readmission_30j: boolean (nullable = true)
 |-- nb_admissions_12mois: integer (nullable = true)
 |-- nb_medicaments: integer (nullable = 

In [0]:
# ========================================
# FEATURES DÉMOGRAPHIQUES
# ========================================
print("\n CATÉGORIE 1 : FEATURES DÉMOGRAPHIQUES\n")
df_fe = df
# 1. Catégories d'âge (plus granulaires)
df_fe = df_fe.withColumn("age_category",
    F.when(F.col("age") < 40, "18-39")
     .when(F.col("age") < 50, "40-49")
     .when(F.col("age") < 60, "50-59")
     .when(F.col("age") < 70, "60-69")
     .when(F.col("age") < 80, "70-79")
     .otherwise("80+"))
# 2. Âge au carré (relation non-linéaire)
df_fe = df_fe.withColumn("age_squared", F.col("age") ** 2)
# 3. Interaction âge × genre
df_fe = df_fe.withColumn("age_gender_interaction",
    F.when(F.col("genre") == "M", F.col("age"))
     .otherwise(F.col("age") * -1))
# 4. Senior flag
df_fe = df_fe.withColumn("is_senior", F.col("age") >= 65)
# 5. Distance hopital catégorisée
df_fe = df_fe.withColumn("distance_category",
    F.when(F.col("distance_hopital_km") < 5, "Très proche")
     .when(F.col("distance_hopital_km") < 20, "Proche")
     .when(F.col("distance_hopital_km") < 50, "Moyen")
     .otherwise("Éloigné"))
# 6. Zone rurale
df_fe = df_fe.withColumn("zone_rurale", F.col("distance_hopital_km") > 30)
print("6 features démographiques créées")
# Features créées: age_category, age_squared, age_gender_interaction, is_senior, distance_category, zone_rurale



 CATÉGORIE 1 : FEATURES DÉMOGRAPHIQUES

6 features démographiques créées


In [0]:
# ========================================
# FEATURES CLINIQUES (COMORBIDITÉS)
# ========================================
print("\n CATÉGORIE 2 : FEATURES CLINIQUES\n")
# 1. Catégories comorbidités
df_fe = df_fe.withColumn("comorbidite_category",
    F.when(F.col("nb_comorbidites") == 0, "Aucune")
     .when(F.col("nb_comorbidites") <= 2, "Légère")
     .when(F.col("nb_comorbidites") <= 4, "Modérée")
     .otherwise("Sévère"))
# 2. Polypathologie flag
df_fe = df_fe.withColumn("polypathologie", F.col("nb_comorbidites") >= 3)

# 3. Charlson Index catégorisé 
# transforme le score numérique brut de l'Indice de Charlson (qui additionne 19 comorbidités avec des poids spécifiques) en groupes de sévérité, classant les patients en quatre catégories de fardeau de comorbidité : aucun, faible, modéré et sévère,
df_fe = df_fe.withColumn("charlson_category",
    F.when(F.col("charlson_index") <= 2, "Faible")
     .when(F.col("charlson_index") <= 4, "Moyen")
     .otherwise("Élevé"))
# 4. Score fragilité (composite) 
# évaluation de la santé des personnes âgées
df_fe = df_fe.withColumn("score_fragilite",
    (F.col("age") / 100.0) * 
    (F.col("nb_comorbidites") + 1) *
    F.when(F.col("insuffisance_cardiaque"), 2.0).otherwise(1.0))
# 5. Combinaisons comorbidités critiques
df_fe = df_fe.withColumn("diabete_hypertension",
    F.col("diabete") & F.col("hypertension"))
df_fe = df_fe.withColumn("cardio_renal",
    F.col("insuffisance_cardiaque") & F.col("maladie_renale"))
df_fe = df_fe.withColumn("trio_mortel",
    F.col("diabete") & F.col("insuffisance_cardiaque") & F.col("maladie_renale"))
# 6. Maladie chronique complexe
df_fe = df_fe.withColumn("maladie_chronique_complexe",
    F.col("diabete") | F.col("hypertension") | F.col("insuffisance_cardiaque"))

print("8 features cliniques créées")
# Features créées: comorbidite_category, polypathologie, charlson_category, score_fragilite, diabete_hypertension, cardio_renal, trio_mortel, maladie_chronique_complexe


 CATÉGORIE 2 : FEATURES CLINIQUES

8 features cliniques créées


In [0]:
# ========================================
# FEATURES MÉDICAMENTS
# ========================================
print("\n CATÉGORIE 3 : FEATURES MÉDICAMENTS\n")
# 1. Polymédication
df_fe = df_fe.withColumn("polymedication", F.col("nb_medicaments") >= 10)
# 2. Catégories médicaments
df_fe = df_fe.withColumn("medicaments_category",
    F.when(F.col("nb_medicaments") <= 3, "Faible")
     .when(F.col("nb_medicaments") <= 7, "Moyen")
     .when(F.col("nb_medicaments") <= 12, "Élevé")
     .otherwise("Très élevé"))
# 3. Ratio médicaments / comorbidités
df_fe = df_fe.withColumn("ratio_medic_comorb",
    F.when(F.col("nb_comorbidites") > 0,
           F.col("nb_medicaments") / F.col("nb_comorbidites"))
     .otherwise(F.col("nb_medicaments").cast("double")))
# 4. Sur-médication (>2 médoc par comorbidité)
df_fe = df_fe.withColumn("sur_medication",
    F.col("ratio_medic_comorb") > 2.0)
print("4 features médicaments créées")
# Features créées: polymedication, medicaments_category, ratio_medic_comorb, sur_medication


 CATÉGORIE 3 : FEATURES MÉDICAMENTS

4 features médicaments créées


In [0]:
# ========================================
# FEATURES TEMPORELLES
# ========================================
print("\n CATÉGORIE 4 : FEATURES TEMPORELLES\n")
# 1. Durée séjour catégorisée
df_fe = df_fe.withColumn("duree_sejour_category",
    F.when(F.col("duree_sejour_jours") <= 2, "Très court")
     .when(F.col("duree_sejour_jours") <= 5, "Court")
     .when(F.col("duree_sejour_jours") <= 10, "Moyen")
     .otherwise("Long"))
# 2. Séjour prolongé
df_fe = df_fe.withColumn("sejour_prolonge", F.col("duree_sejour_jours") > 7)
# 3. Durée séjour log (normalisation)
df_fe = df_fe.withColumn("duree_sejour_log",
    F.log1p(F.col("duree_sejour_jours")))
# 4. Weekend admission - CRÉER LA COLONNE À PARTIR DE date_admission
df_fe = df_fe.withColumn("admission_weekend",
    F.dayofweek(F.col("date_admission")).isin([1, 7]))  
# 5. Mois admission (extrait de date_admission)
df_fe = df_fe.withColumn("mois_admission", 
    F.month(F.col("date_admission")))
# 6. Mois à risque (hiver = infections)
df_fe = df_fe.withColumn("mois_risque_hiver",
    F.col("mois_admission").isin([12, 1, 2]))
# 7. Trimestre
df_fe = df_fe.withColumn("trimestre",
    F.when(F.col("mois_admission").isin([1,2,3]), "Q1")
     .when(F.col("mois_admission").isin([4,5,6]), "Q2")
     .when(F.col("mois_admission").isin([7,8,9]), "Q3")
     .otherwise("Q4"))
# 8. Saison numérique (pour corrélations)
df_fe = df_fe.withColumn("saison_numeric",
    F.when(F.col("saison") == "Printemps", 1)
     .when(F.col("saison") == "Été", 2)
     .when(F.col("saison") == "Automne", 3)
     .otherwise(4))
print("8 features temporelles créées")


 CATÉGORIE 4 : FEATURES TEMPORELLES

8 features temporelles créées


In [0]:
# ========================================
# FEATURES HISTORIQUES
# ========================================
print("\n CATÉGORIE 5 : FEATURES HISTORIQUES\n")
# 1. Réadmission récente
df_fe = df_fe.withColumn("readmission_recente",
    F.col("nb_admissions_12mois") >= 2)
# 2. Hospitalisations fréquentes
df_fe = df_fe.withColumn("hospitalisations_frequentes",
    F.col("nb_admissions_12mois") >= 3)
# 3. Catégories admissions
df_fe = df_fe.withColumn("admissions_category",
    F.when(F.col("nb_admissions_12mois") == 0, "Première")
     .when(F.col("nb_admissions_12mois") == 1, "Deuxième")
     .when(F.col("nb_admissions_12mois") <= 3, "Multiple")
     .otherwise("Chronique"))
# 4. Support social numérique
df_fe = df_fe.withColumn("support_social_numeric",
    F.when(F.col("support_social") == "Faible", 1)
     .when(F.col("support_social") == "Moyen", 2)
     .otherwise(3))
# 5. Isolement social
df_fe = df_fe.withColumn("isolement_social",
    (F.col("support_social") == "Faible") & (F.col("distance_hopital_km") > 20))
# 6. Risque cumulé
df_fe = df_fe.withColumn("risque_cumule",
    F.col("nb_admissions_12mois") * F.col("nb_comorbidites"))
print("6 features historiques créées")
#  Features créées: readmission_recente, hospitalisations_frequentes, admissions_category, support_social_numeric, isolement_social, risque_cumule


 CATÉGORIE 5 : FEATURES HISTORIQUES

6 features historiques créées


In [0]:
# ========================================
# FEATURES GRAVITÉ
# ========================================
print("\n CATÉGORIE 6 : FEATURES GRAVITÉ\n")
# 1. Gravité catégorisée
df_fe = df_fe.withColumn("gravite_category",
    F.when(F.col("score_gravite_admission") <= 3, "Légère")
     .when(F.col("score_gravite_admission") <= 6, "Modérée")
     .otherwise("Sévère"))
# 2. Admission critique
df_fe = df_fe.withColumn("admission_critique",
    (F.col("admission_urgence") == True) & (F.col("score_gravite_admission") >= 7))
print("2 features gravité créées")
# Features créées: gravite_category, admission_critique


 CATÉGORIE 6 : FEATURES GRAVITÉ

2 features gravité créées


In [0]:
# ========================================
# FEATURES INTERACTIONS
# ========================================
print("\n CATÉGORIE 7 : FEATURES INTERACTIONS\n")
# 1. Âge × Gravité
df_fe = df_fe.withColumn("age_gravite_interaction",
    F.col("age") * F.col("score_gravite_admission"))
# 2. Âge × Comorbidités
df_fe = df_fe.withColumn("age_comorbidites_interaction",
    F.col("age") * F.col("nb_comorbidites"))
# 3. Urgence × Gravité
df_fe = df_fe.withColumn("urgence_gravite",
    F.when(F.col("admission_urgence"), F.col("score_gravite_admission"))
     .otherwise(0))
# 4. Polypathologie × Polymédication
df_fe = df_fe.withColumn("poly_poly_interaction",
    F.col("polypathologie").cast("int") * F.col("polymedication").cast("int"))
# 5. Fragilité sociale (âge + support)
df_fe = df_fe.withColumn("fragilite_sociale",
    (F.col("age") > 75) & (F.col("support_social") == "Faible"))
# 6. Risque composite
df_fe = df_fe.withColumn("risque_composite",
    (F.col("age") / 100.0) * 
    F.col("nb_comorbidites") * 
    (F.col("nb_admissions_12mois") + 1))
print("6 features interactions créées")
#  Features créées: age_gravite_interaction, age_comorbidites_interaction, urgence_gravite, poly_poly_interaction, fragilite_sociale, risque_composite


 CATÉGORIE 7 : FEATURES INTERACTIONS

6 features interactions créées


In [0]:
# ========================================
# FEATURES DIAGNOSTICS
# ========================================
print("\n CATÉGORIE 8 : FEATURES DIAGNOSTICS\n")
# Regrouper diagnostics en catégories principales
df_fe = df_fe.withColumn("diagnostic_groupe",
    F.when(F.col("diagnostic_principal").isin([
        "Insuffisance cardiaque", "Infarctus", "AVC"
    ]), "Cardiovasculaire")
     .when(F.col("diagnostic_principal").isin([
        "Pneumonie", "BPCO", "Asthme"
    ]), "Respiratoire")
     .when(F.col("diagnostic_principal").isin([
        "Diabète", "Hyperthyroïdie"
    ]), "Endocrinien")
     .when(F.col("diagnostic_principal").isin([
        "Insuffisance rénale"
    ]), "Rénal")
     .otherwise("Autre"))
# Diagnostic à haut risque
high_risk_diagnoses = ["Insuffisance cardiaque", "Infarctus", "AVC", 
                       "Pneumonie", "Insuffisance rénale"]
df_fe = df_fe.withColumn("diagnostic_haut_risque",
    F.col("diagnostic_principal").isin(high_risk_diagnoses))
print("2 features diagnostics créées")
#  Features créées: diagnostic_groupe, diagnostic_haut_risque


 CATÉGORIE 8 : FEATURES DIAGNOSTICS

2 features diagnostics créées


In [0]:
# ========================================
# FEATURES SERVICE HOSPITALIER
# ========================================
print("\n CATÉGORIE 9 : FEATURES SERVICE\n")
# Services à risque élevé 
# USI Unité de Soins Intensifs
high_risk_services = ["Cardiologie", "Néphrologie", "USI"]
df_fe = df_fe.withColumn("service_haut_risque",
    F.col("service").isin(high_risk_services))
print("1 feature service créée")
# Features créées: service_haut_risque


 CATÉGORIE 9 : FEATURES SERVICE

1 feature service créée


In [0]:
# ========================================
# COMPTAGE TOTAL FEATURES
# ========================================
print("\n" + "="*60)
print("RÉCAPITULATIF FEATURES ENGINEERING")
print("="*60 + "\n")
# Compter nouvelles features
original_cols = set(df.columns)
new_cols = set(df_fe.columns) - original_cols
print(f"Features originales : {len(original_cols)}")
print(f"Nouvelles features : {len(new_cols)}")
print(f"TOTAL : {len(df_fe.columns)}")
print(f"\nListe nouvelles features ({len(new_cols)}) :")
for i, feat in enumerate(sorted(new_cols), 1):
    print(f"   {i:2d}. {feat}")


RÉCAPITULATIF FEATURES ENGINEERING

Features originales : 25
Nouvelles features : 43
TOTAL : 68

Liste nouvelles features (43) :
    1. admission_critique
    2. admission_weekend
    3. admissions_category
    4. age_category
    5. age_comorbidites_interaction
    6. age_gender_interaction
    7. age_gravite_interaction
    8. age_squared
    9. cardio_renal
   10. charlson_category
   11. comorbidite_category
   12. diabete_hypertension
   13. diagnostic_groupe
   14. diagnostic_haut_risque
   15. distance_category
   16. duree_sejour_category
   17. duree_sejour_log
   18. fragilite_sociale
   19. gravite_category
   20. hospitalisations_frequentes
   21. is_senior
   22. isolement_social
   23. maladie_chronique_complexe
   24. medicaments_category
   25. mois_admission
   26. mois_risque_hiver
   27. poly_poly_interaction
   28. polymedication
   29. polypathologie
   30. ratio_medic_comorb
   31. readmission_recente
   32. risque_composite
   33. risque_cumule
   34. saison_num

In [0]:
# ========================================
# SAUVEGARDE DELTA LAKE
# ========================================
print("\n SAUVEGARDE TABLE DELTA...\n")
table_features = "hospital_readmissions_features"
full_table_features = f"{catalog}.{schema_name}.{table_features}"
(df_fe.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_features))
print(f"Table créée : {full_table_features}")
print(f"   Lignes : {df_fe.count():,}")
print(f"   Colonnes : {len(df_fe.columns)}")
# Optimisation
spark.sql(f"OPTIMIZE {full_table_features}")
print("Table optimisée")


SAUVEGARDE TABLE DELTA...

Table créée : workspace.default.hospital_readmissions_features
   Lignes : 69,750
   Colonnes : 68
Table optimisée


In [0]:
# ========================================
# CRÉATION DICTIONNAIRE FEATURES
# ========================================
print("\nCRÉATION DICTIONNAIRE FEATURES \n")
feature_dict = []
for feat in sorted(new_cols):
    # Déterminer catégorie
    if "age" in feat.lower() and "interaction" not in feat.lower():
        category = "Démographique"
    elif any(x in feat.lower() for x in ["diabete", "hyper", "cardio", "renal", "comorb", "charlson"]):
        category = "Clinique"
    elif "medic" in feat.lower():
        category = "Médicaments"
    elif any(x in feat.lower() for x in ["sejour", "saison", "mois", "weekend", "trimestre"]):
        category = "Temporelle"
    elif any(x in feat.lower() for x in ["admission", "support", "isolement", "risque_cumule"]):
        category = "Historique"
    elif "gravite" in feat.lower():
        category = "Gravité"
    elif "interaction" in feat.lower() or "fragilite" in feat.lower():
        category = "Interaction"
    elif "diagnostic" in feat.lower():
        category = "Diagnostic"
    elif "service" in feat.lower():
        category = "Service"
    else:
        category = "Autre"
    feature_dict.append({
        'Feature': feat,
        'Catégorie': category,
        'Type': 'À compléter',
        'Description': 'À compléter',
        'Rationale_Médical': 'À compléter',
        'Source_Littérature': 'À compléter'
    })
dict_df = pd.DataFrame(feature_dict).sort_values(['Catégorie', 'Feature'])
print(f"Dictionnaire créé avec {len(dict_df)} features")
print(f"Réparti en {dict_df['Catégorie'].nunique()} catégories\n")
# Afficher le résumé par catégorie
print("RÉSUMÉ PAR CATÉGORIE:")
print(dict_df['Catégorie'].value_counts().to_string())
print("\n" + "="*60)
print("FEATURE ENGINEERING TERMINÉ")
print("="*60)
# Afficher le tableau complet
print("\nDICTIONNAIRE COMPLET (utilisez le bouton 'Download' ci-dessous):\n")
display(dict_df)


CRÉATION DICTIONNAIRE FEATURES...

Dictionnaire créé avec 43 features
Réparti en 10 catégories

RÉSUMÉ PAR CATÉGORIE:
Catégorie
Autre            8
Temporelle       8
Clinique         6
Historique       6
Interaction      4
Gravité          3
Médicaments      3
Diagnostic       2
Démographique    2
Service          1

FEATURE ENGINEERING TERMINÉ

DICTIONNAIRE COMPLET (utilisez le bouton 'Download' ci-dessous):



Feature,Catégorie,Type,Description,Rationale_Médical,Source_Littérature
distance_category,Autre,À compléter,À compléter,À compléter,À compléter
hospitalisations_frequentes,Autre,À compléter,À compléter,À compléter,À compléter
is_senior,Autre,À compléter,À compléter,À compléter,À compléter
maladie_chronique_complexe,Autre,À compléter,À compléter,À compléter,À compléter
polypathologie,Autre,À compléter,À compléter,À compléter,À compléter
risque_composite,Autre,À compléter,À compléter,À compléter,À compléter
trio_mortel,Autre,À compléter,À compléter,À compléter,À compléter
zone_rurale,Autre,À compléter,À compléter,À compléter,À compléter
age_comorbidites_interaction,Clinique,À compléter,À compléter,À compléter,À compléter
cardio_renal,Clinique,À compléter,À compléter,À compléter,À compléter
